📦 Bloque 1 — Imports

Este bloque importa todas las librerías necesarias para el procesamiento de datos, la conexión a SQL Server, y la lectura de archivos Excel.
- re: Librería para expresiones regulares.
- unicodedata: Usada para normalizar texto eliminando acentos.
- Path: Manipulación de rutas de archivos.
- datetime: Para trabajar con fechas y horas.
- math: Operaciones matemáticas.
- numpy y pandas: Manipulación de matrices y datos en tablas.
- sqlalchemy: Para la conexión con SQL Server.
- openpyxl: Lectura de archivos Excel.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
from typing import Tuple, Optional
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "unidadleasing", "config_unidadleasing.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🔌 Bloque 2 — Conexión a SQL Server

Este bloque establece la conexión con la base de datos SQL Server usando SQLAlchemy y el controlador ODBC.
- create_engine: Establece la conexión con SQL Server usando ODBC.
- connection_string: Define los parámetros de conexión con la base de datos.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 — Parámetros de Entrada
Define los parámetros necesarios para acceder a los archivos Excel y la hoja correspondiente.
- RUTA_CARPETA: Ruta de la carpeta donde se encuentran los archivos Excel.
- SHEET_INDEX: El índice de la hoja dentro del archivo Excel a procesar.
- EXTS: Extensiones de archivos permitidas (en este caso, solo .xlsx).

In [ ]:
# Cambiar el periodo en config/unidadleasing/config_unidadleasing.json
PERIODO      = data.get("periodo")
RUTA_CARPETA = (PROJECT_ROOT / "data" / "unidadleasing" / PERIODO).resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_CARPETA:", RUTA_CARPETA, "exists:", RUTA_CARPETA.exists())

SHEET_INDEX = 0
EXTS        = (".xlsx",)


🔧 Bloque 4 — Normalización de Nombres de Columnas

Este bloque normaliza los nombres de las columnas de los archivos Excel, eliminando acentos y caracteres especiales.
- _strip_accents: Elimina los acentos de las cadenas de texto.
- normalize_name: Normaliza el nombre de las columnas, transformando todo a minúsculas, reemplazando espacios por guiones bajos y quitando caracteres no alfanuméricos.

📊 Bloque 5 — Detección y Eliminación de Footers

Detecta y elimina las filas de "footer" (totalizadores) en los archivos Excel.
- _looks_like_total_text: Función que verifica si una celda contiene palabras clave como "total" o "suma".
- detect_footer_row_index: Identifica el índice de la fila de footer.
- drop_footer_if_present: Elimina la fila de footer si se detecta, y genera un reporte de auditoría.

In [18]:
TOTAL_PATTERNS = ("TOTAL","total", "totales","TOTALES", "suma", "sumatoria")

def _looks_like_total_text(cell) -> bool:
    if cell is None:
        return False
    s = str(cell).strip().lower()
    if len(s) < 3:
        return False
    return any(pat in s for pat in TOTAL_PATTERNS)


def detect_footer_row_index(
    df: pd.DataFrame,
    check_numeric_sum: bool = True
) -> Tuple[Optional[int], Optional[str]]:
    """
    Devuelve (idx_footer, motivo)
      - idx_footer: índice de la fila footer si se detecta; en caso contrario, None
      - motivo: 'keyword_match' si encontró textos tipo 'total' en la última fila,
                'numeric_sum_match' si la última fila ≈ suma de columnas numéricas,
                None si no se detectó footer.
    """
    if df.empty:
        return None, None

    last_idx = df.index[-1]
    last_row = df.loc[last_idx]

    
    if last_row.astype(str).apply(_looks_like_total_text).any():
        return last_idx, "keyword_match"

    
    if check_numeric_sum:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if numeric_cols:
            body = df.iloc[:-1]
            if not body.empty:
                sums = body[numeric_cols].sum(numeric_only=True)
                tol_abs = 1e-6
                tol_rel = 1e-6
                matches = True
                for c in numeric_cols:
                    v_footer = last_row[c]
                    v_sum = sums[c]
                    if pd.isna(v_footer) or pd.isna(v_sum):
                        matches = False
                        break
                    if not math.isclose(float(v_footer), float(v_sum),
                                        rel_tol=tol_rel, abs_tol=tol_abs):
                        matches = False
                        break
                if matches:
                    return last_idx, "numeric_sum_match"

    return None, None


def drop_footer_if_present(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Elimina el footer si existe y retorna (df_limpio, auditoria)
    auditoria = {
        'footer_eliminado': bool,
        'indice_footer': int | None,
        'motivo': str | None,
        'preview_footer': dict | None
    }
    """
    idx, reason = detect_footer_row_index(df)
    audit = {
        "footer_eliminado": False,
        "indice_footer": None,
        "motivo": None,
        "preview_footer": None,
    }
    if idx is None:
        return df, audit

    
    preview = df.loc[idx].to_dict()
    df_clean = df.drop(index=idx)

    audit.update({
        "footer_eliminado": True,
        "indice_footer": int(idx),
        "motivo": reason,
        "preview_footer": preview,
    })
    return df_clean, audit



📅 Bloque 6 — Detección de Periodo (YYYYMM)

Extrae el período YYYYMM desde el nombre del archivo.
- _extract_yyyymm_from_name: Extrae el periodo de las cadenas de texto del nombre del archivo.
- canonicalizar_nombre_archivo: Establece un nombre canónico con el período YYYYMM extraído.

In [19]:
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Extrae YYYYMM desde 'nombre' (sin extensión).
    Soporta: YYYYMM, YYYY[-_/ .]?MM, MM[-_/ .]?YYYY, 'agosto-2025', 'AGO25', etc.
    Lanza ValueError si no puede extraer.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()

    
    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    
    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

   
    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"


    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            # Coincidencia robusta (evita match parcial dentro de otras palabras)
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_nombre_archivo(
    nombre: str,
    prefijo: str = "UNIDAD LEASING - Cesantia - ",
    ext: str = ".xlsx",
    fallback: str = "000000",   # o "timestamp"
) -> str:
    """
    Devuelve '<prefijo>YYYYMM<ext>'.
    Si no logra extraer YYYYMM:
      - fallback='000000'  -> '<prefijo>000000<ext>'
      - fallback='timestamp' -> '<prefijo>YYYYMMDD_HHMMSS<ext>'
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"{prefijo}{yyyymm}{ext}"
    except ValueError as e:
        if fallback == "timestamp":
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            print(f"⚠️ {e}. Usando timestamp: {stamp}")
            return f"{prefijo}{stamp}{ext}"
        else:
            print(f"⚠️ {e}. Usando período genérico: {fallback}")
            return f"{prefijo}{fallback}{ext}"


🔄 Bloque 7 — Lectura y Preprocesamiento de Archivos Excel

Lee los archivos Excel, elimina filas vacías, normaliza los nombres de las columnas, y genera el DataFrame final.
- read_one_excel: Lee un archivo Excel y limpia las columnas.
- iter_excels: Itera sobre todos los archivos Excel en una carpeta.
- drop_footer_if_present: Elimina el footer y genera un reporte.

In [20]:
def read_one_excel(path: Path, sheet_index: int) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet_index, engine="openpyxl")
    df.columns = [fun.normalize_name(c) for c in df.columns]
    df = df.dropna(how="all")
    return df


def iter_excels(folder: Path, exts=(".xlsx",)):
    for p in sorted(folder.glob("*")):
        if p.suffix.lower() in exts and p.is_file():
            yield p

registros = []            # Para el reporte de auditoría por archivo
footers_extraidos = []    # Opcional: guardamos la fila footer para auditoría fina
dfs_limpios = []          # DataFrames listos para concatenar

for file_path in iter_excels(RUTA_CARPETA, EXTS):
    df_raw = read_one_excel(file_path, SHEET_INDEX)
    df_clean, audit = drop_footer_if_present(df_raw)


    nombre_canonico = canonicalizar_nombre_archivo(file_path.name)
    df_clean["NOMBRE_ARCHIVO"] = nombre_canonico

 
    reg = {
        "archivo": file_path.name,
        "footer_eliminado": audit["footer_eliminado"],
        "indice_footer": audit["indice_footer"],
        "motivo": audit["motivo"],
    }
    registros.append(reg)

   
    if audit["footer_eliminado"] and audit["preview_footer"] is not None:
        footer_row = dict(audit["preview_footer"])
        footer_row["__origen_archivo__"] = file_path.name
        footers_extraidos.append(footer_row)

    dfs_limpios.append(df_clean)


if not dfs_limpios:
    raise RuntimeError("No se encontraron archivos Excel válidos en la carpeta.")

df_final = pd.concat(dfs_limpios, ignore_index=True)


reporte_footers = pd.DataFrame(registros)


df_footers = pd.DataFrame(footers_extraidos) if footers_extraidos else pd.DataFrame()


print("=== REPORTE DE FOOTERS ELIMINADOS ===")
print(reporte_footers)

if not df_footers.empty:
    print("\n=== PREVIEW DE FOOTERS EXTRAÍDOS (primeras 4 filas) ===")
    display(df_footers.head(4))
else:
    print("\nNo se extrajeron filas de footer.")

print("=== RESUMEN FINAL ===")
print(f"Archivos procesados: {len(reporte_footers)}")
print(f"Total de filas finales (sin footers): {len(df_final):,}")
print(f"Total de filas de footer eliminadas: {len(df_footers):,}")

=== REPORTE DE FOOTERS ELIMINADOS ===
                                        archivo  footer_eliminado  \
0  Cesantia ULH POL 8846590 Noviembre 2025.xlsx              True   
1  Cesantia ULH POL 8846591 Noviembre 2025.xlsx              True   
2  Cesantia ULH POL 8846592 Noviembre 2025.xlsx              True   
3  Cesantia ULH POL 8846593 Noviembre 2025.xlsx              True   

   indice_footer         motivo  
0           2013  keyword_match  
1             36  keyword_match  
2           3142  keyword_match  
3            702  keyword_match  

=== PREVIEW DE FOOTERS EXTRAÍDOS (primeras 4 filas) ===


,unnamed_0,operacion,oper_unidad,tac,ctr_adm,rut_deudor,dv,deudor,tipo_vivienda,direccion,...,valor_cupon_s_seg,seg_inc,seg_dgvm,seg_ces,seg_ces_afecto,seg_ces_iva,tot_cupon_c_seg,celular,mail,__origen_archivo__
0,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,895.983069,752.926949,143.056120,15794.2657,NaN,NaN,Cesantia ULH POL 8846590 Noviembre 2025.xlsx
1,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,17.447500,14.661765,NaN,316.3942,NaN,NaN,Cesantia ULH POL 8846591 Noviembre 2025.xlsx
2,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,568.8009,225.9345,1276.584600,1072.760168,203.824432,22207.6301,NaN,NaN,Cesantia ULH POL 8846592 Noviembre 2025.xlsx
3,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111.0041,43.8741,179.829600,151.117311,28.712289,4717.8835,NaN,NaN,Cesantia ULH POL 8846593 Noviembre 2025.xlsx


=== RESUMEN FINAL ===
Archivos procesados: 4
Total de filas finales (sin footers): 5,893
Total de filas de footer eliminadas: 4


📥 Bloque 8 — Mapeo y Selección de Columnas

Este bloque selecciona las columnas del DataFrame y las mapea a nuevas columnas para su posterior procesamiento.
- pick: Función que selecciona columnas específicas del DataFrame.
- Construcción del nuevo DataFrame: Se construye el nuevo df_can a partir de las columnas seleccionadas.

In [21]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None] * len(df), index=df.index)


guia                = pick(df_final, "unnamed_0")
operacion           = pick(df_final, "operacion", "operación", "Operacion")
oper_unidad         = pick(df_final, "oper_unidad", "operacion_unidad", "operacionunidad")
rut                 = pick(df_final, "rut_deudor", "rut")
dv                  = pick(df_final, "dv", "dv_deudor")
tipo_vivienda       = pick(df_final, "tipo_vivienda", "vivienda")
comuna              = pick(df_final, "comuna")
poliza              = pick(df_final, "nro_poliza", "nro_póliza", "poliza")
prima_ces           = pick(df_final, "prima_ces")
fecha_ctto          = pick(df_final, "fecha_ctto")
fecha_vcmto         = pick(df_final, "fecha_vcmto")
fecha_carga         = pick(df_final, "fecha_carga", "fechacarga")
fecha_nac           = pick(df_final, "fecha_nac", "fechanac", "fecha_nacimiento")
primer_vcmto        = pick(df_final, "primer_vcmto")
prima_bruta         = pick(df_final, "seg_ces")
monto_asegurado     = pick(df_final, "tot_cupon_c_seg")
nombre_archivo      = pick(df_final, "nombre_archivo", "NOMBRE_ARCHIVO")


df_can = pd.DataFrame({
    "Guia": guia,
    "Operacion": operacion,
    "Oper_Unidad": oper_unidad,
    "RUT": rut,
    "DV": dv,
    "Tipo_Vivienda": tipo_vivienda,
    "Comuna": comuna,
    "Poliza": poliza,
    "Prima_Ces": prima_ces,
    "Fecha_Ctto": fecha_ctto,
    "Fecha_Vcmto": fecha_vcmto,
    "Fecha_Carga": fecha_carga,
    "Fecha_Nac": fecha_nac,
    "Primer_Vcmto": primer_vcmto,
    "Prima_Bruta_PPI": prima_bruta,
    "Monto_Asegurado": monto_asegurado,
    "NOMBRE_ARCHIVO": nombre_archivo,
})

df_can.head()

,Guia,Operacion,Oper_Unidad,RUT,DV,Tipo_Vivienda,Comuna,Poliza,Prima_Ces,Fecha_Ctto,Fecha_Vcmto,Fecha_Carga,Fecha_Nac,Primer_Vcmto,Prima_Bruta_PPI,Monto_Asegurado,NOMBRE_ARCHIVO
0,1,00000103,00000103,25736304.0,K,DEPARTAMENTO,LOS ANDES,8382351.0,0.0595,2024-05-30,2044-06-10,2025-04-15 15:15:46,1993-06-05,10-07-2024,0.3588,7.0958,UNIDAD LEASING - Cesantia - 202511.xlsx
1,2,2023126287,2023126287,26276745.0,0,DEPARTAMENTO,RENCA,8382351.0,0.0595,2023-12-31,2044-03-10,2024-01-11 10:11:35,1996-10-26,10-04-2024,0.3697,6.6688,UNIDAD LEASING - Cesantia - 202511.xlsx
2,3,2023126302,2023126302,25856878.0,8,DEPARTAMENTO,EL BOSQUE,8382351.0,0.0595,2023-12-31,2044-03-10,2024-01-11 10:11:35,1988-10-04,10-04-2024,0.3920,7.1473,UNIDAD LEASING - Cesantia - 202511.xlsx
3,4,2024016372,2024016372,14323150.0,K,DEPARTAMENTO,EL BOSQUE,8382351.0,0.0595,2024-01-31,2044-04-10,2024-02-07 11:39:23,1977-05-04,10-05-2024,0.4516,8.1725,UNIDAD LEASING - Cesantia - 202511.xlsx
4,5,2024016384,2024016384,18967318.0,3,CASA,MOLINA,8382351.0,0.0595,2024-01-31,2044-02-10,2024-02-07 11:39:23,1995-08-16,10-03-2024,0.6447,11.3130,UNIDAD LEASING - Cesantia - 202511.xlsx


🔢 Bloque 9 — Limpieza de Operaciones

Este bloque limpia las columnas Operacion y Oper_Unidad para generar nuevas columnas sin caracteres no numéricos, lo que permite que se conviertan correctamente a tipo BIGINT en SQL.
- _only_digits_series: Función que elimina todo lo que no sea un dígito (como letras, espacios, etc.).
- Operacion2 y Oper_Unidad2: Nuevas columnas limpias que contienen solo los números de las columnas Operacion y Oper_Unidad.

In [22]:
def _only_digits_series(s: pd.Series) -> pd.Series:
    out = s.astype(str).str.replace(r"\D+", "", regex=True)
    out = out.replace("", np.nan)
    return out


if "Operacion" not in df_can.columns:
    df_can["Operacion"] = np.nan
if "Oper_Unidad" not in df_can.columns:
    df_can["Oper_Unidad"] = np.nan

df_can["Operacion2"]    = _only_digits_series(df_can["Operacion"])
df_can["Oper_Unidad2"]  = _only_digits_series(df_can["Oper_Unidad"])

m_op  = df_can["Operacion"].astype(str)   != df_can["Operacion2"].astype(str)
m_uni = df_can["Oper_Unidad"].astype(str) != df_can["Oper_Unidad2"].astype(str)

print("=== RESUMEN LIMPIEZA NUMÉRICA ===")
print(f"Filas afectadas en Operacion  -> {int(m_op.sum()):,}")
print(f"Filas afectadas en Oper_Unidad -> {int(m_uni.sum()):,}")

ej_op  = df_can.loc[m_op,  ["Operacion", "Operacion2"]].head(5)
ej_uni = df_can.loc[m_uni, ["Oper_Unidad", "Oper_Unidad2"]].head(5)

if not ej_op.empty:
    print("\nEjemplos Operacion → Operacion2:")
    display(ej_op)
if not ej_uni.empty:
    print("\nEjemplos Oper_Unidad → Oper_Unidad2:")
    display(ej_uni)

=== RESUMEN LIMPIEZA NUMÉRICA ===
Filas afectadas en Operacion  -> 767
Filas afectadas en Oper_Unidad -> 767

Ejemplos Operacion → Operacion2:


,Operacion,Operacion2
181,1995120190A003,1995120190003
182,2007120574A003,2007120574003
183,2010120725A003,2010120725003
184,2012030204A003,2012030204003
185,2013050406A003,2013050406003



Ejemplos Oper_Unidad → Oper_Unidad2:


,Oper_Unidad,Oper_Unidad2
181,1995120190A003,1995120190003
182,2007120574A003,2007120574003
183,2010120725A003,2010120725003
184,2012030204A003,2012030204003
185,2013050406A003,2013050406003


🗓️ Bloque 10 — Extracción de la Fecha de Recibo desde el Nombre del Archivo

Este bloque toma el nombre del archivo y extrae el período YYYYMM para asignarlo a una nueva columna llamada MES_RECAUDACION. 

Esto nos permite organizar los datos por mes y año, basándonos en el nombre del archivo que los contiene.

In [23]:
def extraer_yyyymm_de_nombre_archivo(df: pd.DataFrame, nombre_columna: str = "NOMBRE_ARCHIVO") -> pd.Series:
    """
    Extrae el YYYYMM de la columna `Nombre_de_archivo` usando el mismo patrón de extracción.
    """
    return df[nombre_columna].str.extract(r"(20\d{2})(0[1-9]|1[0-2])", expand=False).apply(lambda x: ''.join(x), axis=1)

# Generamos la nueva columna MES_RECAUDACION a partir de Nombre_Archivo
df_can["MES_RECAUDACION"] = extraer_yyyymm_de_nombre_archivo(df_can)

# (Opcional) Validar si se extrajo correctamente el YYYYMM y mostrar ejemplos
if df_can["MES_RECAUDACION"].isnull().any():
    print("⚠️ Algunos valores de MES_RECAUDACION no pudieron ser extraídos.")
    
# Muestra rápida de las primeras filas para revisión
df_can[["NOMBRE_ARCHIVO", "MES_RECAUDACION"]].head(3)

,NOMBRE_ARCHIVO,MES_RECAUDACION
0,UNIDAD LEASING - Cesantia - 202511.xlsx,202511
1,UNIDAD LEASING - Cesantia - 202511.xlsx,202511
2,UNIDAD LEASING - Cesantia - 202511.xlsx,202511


⚙️ Bloque 11 — Tipado y Conversión de Datos

Este bloque convierte las columnas de datos al tipo correcto. 

Si una columna debe ser entera, la convertimos en tipo INT; si es una fecha, la transformamos a formato de fecha DATE. 

Esto asegura que todos los datos estén listos para ser cargados correctamente en la base de datos.

In [27]:
INT_COLS    = ["Guia", "RUT", "Poliza", "MES_RECAUDACION"]
BIGINT_COLS = ["Operacion2", "Oper_Unidad2"]
FLOAT_COLS  = ["Prima_Ces", "Prima_Bruta_PPI", "Monto_Asegurado"]
STR_COLS    = ["Operacion", "Oper_Unidad", "Tipo_Vivienda", "Comuna", "NOMBRE_ARCHIVO"]
DV_COL      = "DV"
DATE_COLS   = ["Fecha_Ctto", "Fecha_Vcmto", "Fecha_Carga", "Fecha_Nac", "Primer_Vcmto"]




def _to_num_series(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """
    Convierte las columnas de fechas de formato object a formato datetime (YYYY-MM-DD).
    Si la conversión falla, reemplaza con NaT (Not a Time).
    """
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )


if "Operacion" in df_can.columns:
    df_can["Operacion"] = df_can["Operacion"].astype("string").str.strip().str.slice(0, 50)

if "Oper_Unidad" in df_can.columns:
    df_can["Oper_Unidad"] = df_can["Oper_Unidad"].astype("string").str.strip().str.slice(0, 50)

if "Tipo_Vivienda" in df_can.columns:
    df_can["Tipo_Vivienda"] = df_can["Tipo_Vivienda"].astype("string").str.strip().str.slice(0, 50)

if "Comuna" in df_can.columns:
    df_can["Comuna"] = df_can["Comuna"].astype("string").str.strip().str.slice(0, 50)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 50)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)


criticas = ["Operacion","RUT","Poliza","NOMBRE_ARCHIVO"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")


cols_sql = [
    "Guia", "Operacion", "Oper_Unidad", "RUT", "DV", "Tipo_Vivienda", "Comuna", "Poliza",
    "Prima_Ces", "Fecha_Ctto", "Fecha_Vcmto", "Fecha_Carga", "Fecha_Nac", "Primer_Vcmto",
    "Prima_Bruta_PPI", "Monto_Asegurado", "NOMBRE_ARCHIVO", "Operacion2", "Oper_Unidad2", "MES_RECAUDACION"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_5464\3776583352.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_5464\3776583352.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_5464\3776583352.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated 

✅ dtypes DESPUÉS:
 Guia                        Int64
Operacion          string[python]
Oper_Unidad        string[python]
RUT                         Int64
DV                         object
Tipo_Vivienda      string[python]
Comuna             string[python]
Poliza                      Int64
Prima_Ces                 float64
Fecha_Ctto         datetime64[ns]
Fecha_Vcmto        datetime64[ns]
Fecha_Carga        datetime64[ns]
Fecha_Nac          datetime64[ns]
Primer_Vcmto       datetime64[ns]
Prima_Bruta_PPI           float64
Monto_Asegurado           float64
NOMBRE_ARCHIVO     string[python]
Operacion2                  Int64
Oper_Unidad2                Int64
MES_RECAUDACION             Int64
dtype: object

🔎 Nulos en columnas críticas:
 - Operacion: 0 nulos
 - RUT: 0 nulos
 - Poliza: 0 nulos
 - NOMBRE_ARCHIVO: 0 nulos

📦 df_sql listo con columnas: ['Guia', 'Operacion', 'Oper_Unidad', 'RUT', 'DV', 'Tipo_Vivienda', 'Comuna', 'Poliza', 'Prima_Ces', 'Fecha_Ctto', 'Fecha_Vcmto', 'Fecha_Carga'

In [29]:
#Parche para las Fechas de truncamientos DateTime a Date
MIN_DT = pd.Timestamp("1753-01-01")
MAX_DT = pd.Timestamp("9999-12-31")

date_cols = DATE_COLS

for col in date_cols:
    if col not in df_sql.columns:
        continue
    
    # Ya deberían ser datetime por el paso 1; reforzamos:
    df_sql[col] = pd.to_datetime(df_sql[col], errors="coerce")

    # Cualquier cosa fuera de rango → NaT
    mask_out = df_sql[col].notna() & ((df_sql[col] < MIN_DT) | (df_sql[col] > MAX_DT))
    if mask_out.any():
        print(f"⚠️ {mask_out.sum()} filas fuera de rango en {col}, se pondrán en NULL")
        df_sql.loc[mask_out, col] = pd.NaT

    # Convertimos a tipo date (python date) para encajar perfecto con SQL DATE
    df_sql[col] = df_sql[col].dt.date

🧾 Bloque 12 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [30]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - UNIDAD LEASING - Cesantia - 202511.xlsx: 5893 filas


🔄 Bloque 13 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [31]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:  
       
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

    
        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        
        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para NOMBRE_ARCHIVO = 'UNIDAD LEASING - Cesantia - 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 5893 filas para NOMBRE_ARCHIVO = 'UNIDAD LEASING - Cesantia - 202511.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - UNIDAD LEASING - Cesantia - 202511.xlsx: 5893 filas cargadas (archivo nuevo).
